In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
from dash.exceptions import PreventUpdate
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#FIXED: imported CRUD_Python_Module and AnimalShelter class
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

#FIXED: added aacuser username/password
username = "aacuser"
password = "mikka17"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

#FIXED: added Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIXED: added logo image tag + unique identifier into app.layout
app.layout = html.Div([

    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '100px', 'float': 'left', 'margin-right': '20px'}
        ),
        html.Div([
            html.H1('Grazioso Salvare Rescue Animal Dashboard',
                    style={'margin': '0', 'padding-top': '10px'}),
            html.P('Dashboard developed by Joseph M. Opheim | CS-340 | June 2026',
                   style={'margin': '0', 'color': 'gray', 'font-style': 'italic'})
        ])
    ], style={'overflow': 'hidden', 'padding': '10px', 'border-bottom': '2px solid #ccc'}),

    html.Hr(),

    #FIXED: added interactive filtering options (radio buttons for rescue type)
    html.Div([
        html.H3('Filter by Rescue Type'),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'WR'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'MR'},
                {'label': 'Disaster or Individual Tracking', 'value': 'DIT'},
                {'label': 'Reset (Show All)', 'value': 'Reset'}
            ],
            value='Reset',
            inline=True,
            style={'font-size': '16px', 'gap': '20px'}
        )
    ], style={'padding': '15px', 'background-color': '#f9f9f9', 'border-radius': '8px', 'margin': '10px'}),

    html.Hr(),

    #FIXED: added data table features - pagination, sorting, row/column selection
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),
        page_size=10,
        page_action='native',
        sort_action='native',
        sort_mode='single',
        row_selectable='single',
        selected_rows=[],
        column_selectable='single',
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '8px',
            'font-size': '13px',
            'whiteSpace': 'normal',
            'height': 'auto'
        },
        style_header={
            'backgroundColor': '#003366',
            'color': 'white',
            'fontWeight': 'bold'
        },
        style_data_conditional=[
            {'if': {'row_index': 'odd'}, 'backgroundColor': '#f2f2f2'}
        ]
    ),

    html.Br(),
    html.Hr(),

    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'flex': '1', 'padding': '10px'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'flex': '1', 'padding': '10px'}
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    #FIXED: added MongoDB queries for each rescue type filter

    if filter_type == 'WR':  # Water Rescue
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': [
                'Labrador Retriever Mix',
                'Chesapeake Bay Retriever',
                'Newfoundland'
            ]},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    elif filter_type == 'MR':  # Mountain or Wilderness Rescue
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': [
                'German Shepherd',
                'Alaskan Malamute',
                'Old English Sheepdog',
                'Siberian Husky',
                'Rottweiler'
            ]},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }

    elif filter_type == 'DIT':  # Disaster or Individual Tracking
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': [
                'Doberman Pinscher',
                'German Shepherd',
                'Golden Retriever',
                'Bloodhound',
                'Rottweiler'
            ]},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }

    else:  # Reset - return all records
        query = {}

    filtered_df = pd.DataFrame.from_records(db.read(query))

    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')


#FIXED: added pie chart that updates based on filtered table data
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    if viewData is None:
        raise PreventUpdate

    dff = pd.DataFrame(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return [html.P('No data available for selected filter.',
                       style={'color': 'gray', 'text-align': 'center'})]

    fig = px.pie(
        dff,
        names='breed',
        title='Breed Distribution of Filtered Results',
        hole=0.3
    )
    fig.update_traces(textposition='inside', textinfo='percent+label')

    return [dcc.Graph(figure=fig)]


# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    if viewData is None:
        return

    dff = pd.DataFrame.from_dict(viewData)

    #FIXED: original code crashed on empty selection list, now defaults to row 0
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),
                # Marker with tool tip and popup
                # Column 13 and 14 define the grid-coordinates for the map
                # Column 4 defines the breed for the animal
                # Column 9 defines the name of the animal
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app,
# the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(debug=True)

Dash app running on https://concertguest-cityaudio-3000.codio.io/proxy/8050/
